In [13]:
import anthropic
import duckdb
import os
from pathlib import Path
from dotenv import load_dotenv


In [14]:
# load environment variables from .env file
load_dotenv()

True

In [15]:
# connections
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
conn = duckdb.connect(str(Path("olist_pipeline/dev.duckdb")))

In [16]:
print("Setup complete")
print(f"Claude client ready: {client is not None}")
print(f"DuckDB connected: {conn is not None}")

Setup complete
Claude client ready: True
DuckDB connected: True


In [17]:
def get_top_competed_product():
    """Get the product with the most seller competition"""
    result = conn.execute("""
        select 
            product_id, 
            category_name_english, 
            max(seller_count) as seller_count
        from mart_seller_value_score
        group by product_id, category_name_english
        order by seller_count desc, sum(order_count) desc
        limit 1
    """).fetchone()
    return result


In [18]:
product = get_top_competed_product()
print(product)

('d285360f29ac7fd97640bf0baef03de0', 'watches_gifts', 8)


In [19]:
print(f"\nTop competed product:")
print(f"Category:  {product[1]}")
print(f"Sellers:   {product[2]}")
print(f"ProductID: {product[0]}")


Top competed product:
Category:  watches_gifts
Sellers:   8
ProductID: d285360f29ac7fd97640bf0baef03de0


In [20]:
def get_seller_rankings(product_id: str) -> list:
    """Get all seller rankings for a specific product"""
    results = conn.execute("""
        select
            seller_city,
            seller_state,
            avg_price,
            avg_delivery_days,
            avg_review_score,
            delayed_orders,
            on_time_orders,
            price_rank,
            delivery_rank,
            quality_rank
        from mart_seller_value_score
        where product_id = ?
        order by price_rank
    """, [product_id]).fetchall()

    return [
        {
            "city": r[0],
            "state": r[1],
            "avg_price": r[2],
            "avg_delivery_days": r[3],
            "avg_review_score": r[4],
            "delayed_orders": r[5],
            "on_time_orders": r[6],
            "price_rank": r[7],
            "delivery_rank": r[8],
            "quality_rank": r[9]
        }
        for r in results
    ]

In [21]:
sellers = get_seller_rankings(product[0])
print(f"\nSeller rankings ({len(sellers)} sellers):")
for s in sellers:
    print(f"  {s['city']}: price {s['avg_price']} BRL | "
          f"delivery {s['avg_delivery_days']} days | "
          f"rating {s['avg_review_score']} stars")


Seller rankings (7 sellers):
  campinas: price 184.87 BRL | delivery 16.9 days | rating 3.16 stars
  cotia: price 271.0 BRL | delivery 5.4 days | rating 3.88 stars
  ribeirao preto: price 273.5 BRL | delivery 13.3 days | rating 3.5 stars
  guariba: price 316.59 BRL | delivery 17.4 days | rating 4.31 stars
  pradopolis: price 322.51 BRL | delivery 15.3 days | rating 4.2 stars
  mogi das cruzes: price 341.5 BRL | delivery 5.0 days | rating 5.0 stars
  sumare: price 348.8 BRL | delivery 14.4 days | rating 4.32 stars


In [22]:
def generate_seller_brief(product_id: str, category: str, seller_count: int, sellers: list) -> str:
    """Generate AI brief for marketplace operator"""

    sellers_text = "\n".join([
        f"- {s['city']}, {s['state']}: "
        f"Price {s['avg_price']} BRL (rank #{s['price_rank']}), "
        f"Delivery {s['avg_delivery_days']} days (rank #{s['delivery_rank']}), "
        f"Rating {s['avg_review_score']} stars (rank #{s['quality_rank']}), "
        f"{s['on_time_orders']} on-time / {s['delayed_orders']} delayed orders"
        for s in sellers
    ])

    prompt = f"""You are a marketplace intelligence analyst helping operations teams understand seller competition.

A product in the {category} category has {seller_count} competing sellers:

{sellers_text}

Write a concise seller intelligence brief for a marketplace operator. Include:
1. The key competitive tension — price vs quality vs speed tradeoff
2. Which seller offers best overall value for buyers and why
3. Which seller is most at risk of damaging platform reputation and why
4. One specific recommendation for the marketplace operator

Plain English only. Use actual numbers. Write exactly 4 paragraphs.
Each paragraph maximum 50 words. No bullet points. No headers."""

    # Count input tokens exactly
    input_tokens = client.messages.count_tokens(
        model="claude-sonnet-4-6",
        messages=[{"role": "user", "content": prompt}]
    ).input_tokens

    # Estimate output tokens
    estimated_words = 4 * 50
    estimated_tokens = int(estimated_words * 1.3)
    safety_margin = int(estimated_tokens * 0.40)
    output_tokens = estimated_tokens + safety_margin

    print(f"\nInput tokens (exact):      {input_tokens}")
    print(f"Output tokens (estimated): {output_tokens}")
    print(f"Estimated cost:            ${((input_tokens * 3) + (output_tokens * 15)) / 1_000_000:.6f}")

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=output_tokens,
        messages=[{"role": "user", "content": prompt}]
    )

    actual_cost = ((response.usage.input_tokens * 3) + (response.usage.output_tokens * 15)) / 1_000_000
    print(f"Actual input tokens:       {response.usage.input_tokens}")
    print(f"Actual output tokens:      {response.usage.output_tokens}")
    print(f"Actual cost:               ${actual_cost:.6f}")

    return response.content[0].text

In [23]:
print("\nGenerating seller intelligence brief...")
brief = generate_seller_brief(product[0], product[1], product[2], sellers)
print("\n" + "=" * 50)
print("SELLER INTELLIGENCE BRIEF")
print("=" * 50)
print(brief)
print("=" * 50)


Generating seller intelligence brief...

Input tokens (exact):      545
Output tokens (estimated): 364
Estimated cost:            $0.007095
Actual input tokens:       545
Actual output tokens:      313
Actual cost:               $0.006330

SELLER INTELLIGENCE BRIEF
The market spans 184.87 to 348.80 BRL, revealing a clear tradeoff: campinas wins on price but loses on speed and quality, while mogi das cruzes charges 85% more yet delivers in 5 days with a perfect 5.0 rating. Buyers must choose between saving money or getting reliability.

Mogi das cruzes offers the best overall buyer value despite its 341.50 BRL price. It delivers fastest at 5.0 days, holds a perfect 5.0 rating, and has zero delayed orders from 4 shipments. Cotia is a strong runner-up at 271.00 BRL with 5.4-day delivery and no delays.

Campinas poses the greatest reputational risk. At 184.87 BRL it attracts volume, with 55 total orders, but delivers in 16.9 days, carries a 3.16 rating, and has 1 delayed order in its hist